# AC-MOT v10_p4 — FP16 Fair Benchmark + Deployment Validation

This notebook preserves v10_p3 and runs the separate v10_p4 FP16 fair-comparison protocol.
It also hardens Google Drive remount/staging because Colab Drive mounts can occasionally disconnect.

In [ ]:
# CELL 1 — GPU + INSTALL + ROBUST DRIVE MOUNT
!nvidia-smi
!pip install -q ultralytics==8.3.200 motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git
import os, subprocess, sys, torch, ultralytics
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'CUDA is not available'
assert 'T4' in torch.cuda.get_device_name(0), f'Expected Tesla T4, found {torch.cuda.get_device_name(0)}'
assert os.path.isdir('/content/drive/MyDrive'), 'Google Drive mount failed'
print('PRE-FLIGHT OK')


In [ ]:
# CELL 2 — FRESH PUBLIC CLONE (NO TOKEN REQUIRED)
import pathlib, shutil, subprocess
REPO='/content/ACMOT-Codex-V10Style-Portable'
REPO_URL='https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git'
if pathlib.Path(REPO).exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('Repo ready:', REPO)
print('Commit:', commit)
assert pathlib.Path(REPO,'fair_benchmark_v10_p4.py').exists()
assert pathlib.Path(REPO,'drone_runtime_v10_p4.py').exists()


In [ ]:
# CELL 3 — VERIFY + STAGE DATASET LOCALLY WITH DRIVE-RETRY
from pathlib import Path
import shutil, time, pandas as pd
from tqdm.auto import tqdm
from google.colab import drive
DRIVE_DATASET=Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
LOCAL=Path('/content/visdrone_v10_p4_local/VisDrone2019-MOT-test-dev')
SEQ=DRIVE_DATASET/'sequences'; ANN=DRIVE_DATASET/'annotations'
def remount_drive():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(1)
    drive.mount('/content/drive', force_remount=True)
def drive_retry(fn, label, attempts=3):
    last=None
    for attempt in range(1, attempts+1):
        try:
            return fn()
        except OSError as e:
            last=e
            print(f'Drive I/O failed during {label} (attempt {attempt}/{attempts}): {e}')
            if attempt < attempts:
                remount_drive(); time.sleep(2)
    raise last
assert drive_retry(lambda: SEQ.exists(), 'sequences path check'), f'Missing sequences folder: {SEQ}'
assert drive_retry(lambda: ANN.exists(), 'annotations path check'), f'Missing annotations folder: {ANN}'
seqs=drive_retry(lambda: sorted([p.name for p in SEQ.iterdir() if p.is_dir()]), 'sequence listing')
assert len(seqs)==17, f'Expected 17 sequences, found {len(seqs)}'
manifest=[]
for s in seqs:
    frames=drive_retry(lambda s=s: sorted((SEQ/s).glob('*.jpg')), f'frame listing {s}')
    gt=drive_retry(lambda s=s: pd.read_csv(ANN/f'{s}.txt',header=None), f'GT read {s}')
    gt_max=int(gt.iloc[:,0].max())
    assert len(frames)==gt_max, f'Mismatch {s}: frames={len(frames)}, gt_max={gt_max}'
    manifest.append((s,len(frames)))
if LOCAL.exists(): shutil.rmtree(LOCAL)
(LOCAL/'sequences').mkdir(parents=True); (LOCAL/'annotations').mkdir(parents=True)
total=sum(n for _,n in manifest)+17
p=tqdm(total=total,desc='Drive -> /content',dynamic_ncols=True)
for s,n in manifest:
    dst=LOCAL/'sequences'/s; dst.mkdir()
    frames=drive_retry(lambda s=s: sorted((SEQ/s).glob('*.jpg')), f'frame relist {s}')
    for fp in frames:
        target=dst/fp.name
        drive_retry(lambda fp=fp,target=target: shutil.copy2(fp,target), f'copy {s}/{fp.name}')
        p.update(1)
    gt_src=ANN/f'{s}.txt'; gt_dst=LOCAL/'annotations'/f'{s}.txt'
    drive_retry(lambda gt_src=gt_src,gt_dst=gt_dst: shutil.copy2(gt_src,gt_dst), f'copy GT {s}')
    p.update(1)
p.close()
print('17/17 sequences verified and staged locally')
print('Local dataset ready:', LOCAL)


In [ ]:
# CELL 4 — FP16 FAIR BENCHMARK + OFFICIAL TRACKEVAL
from datetime import datetime
from pathlib import Path
import subprocess, sys, shutil, torch
from ultralytics import YOLO
WEIGHTS=Path('/content/yolov8n.pt')
if not WEIGHTS.exists():
    print('Downloading official YOLOv8n weights...')
    m=YOLO('yolov8n.pt')
    candidate=Path('yolov8n.pt').resolve()
    if candidate != WEIGHTS and candidate.exists(): shutil.copy2(candidate, WEIGHTS)
    del m
assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS}'
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
OUT=Path('/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR') / ('codex_v10_p4_fp16_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
cmd=[sys.executable, f'{REPO}/fair_benchmark_v10_p4.py', '--dataset', str(LOCAL), '--weights', str(WEIGHTS), '--output', str(OUT), '--run-trackeval']
print('Running:', ' '.join(cmd))
proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(proc.stdout)
log_root=Path('/content/v10_p4_logs'); log_root.mkdir(exist_ok=True)
(log_root/'LAST_V10_P4_CONSOLE_LOG.txt').write_text(proc.stdout or '',encoding='utf-8')
if proc.returncode != 0:
    print('===== LAST 120 LINES OF REAL ERROR =====')
    print('\n'.join((proc.stdout or '').splitlines()[-120:]))
    raise RuntimeError(f'v10_p4 benchmark failed with exit code {proc.returncode}')
print('FINAL RESULT FOLDER:', OUT)


In [ ]:
# CELL 5 — OPTIONAL DEPLOYMENT SIMULATION AFTER BENCHMARK
VIDEO='/content/drive/MyDrive/your_flight_video.mp4'
DEPLOY_OUT='/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR/drone_simulation'
# !python {REPO}/drone_runtime_v10_p4.py --source "{VIDEO}" --weights /content/yolov8n.pt --output-dir "{DEPLOY_OUT}" --save-video
